# WavMamba — WiFi-CSI HAR on UT-HAR & NTU-Fi

Public code accompanying the paper. Trains **WavMamba** — a multi-branch
(one branch per Haar DWT subband) CNN + bidirectional-Mamba model with adaptive
late fusion — for WiFi-CSI human activity recognition on **UT-HAR** and **NTU-Fi**.

Architecture (fixed): Haar `{HL, LH}` branches, attentive statistics pooling,
no stem GroupNorm, per-channel gate fusion.

Normalization is controlled by two orthogonal flags:
- `PRENORM` — pre-norm on raw (before DWT): `sensefi` (UT-HAR min-max / NTU-Fi (x-42.32)/4.98) | `none` (no raw pre-normalization).
- `Z_GRAN`  — z-norm after DWT (always on): `perpos` (per-position `(C,T2,F2)`) | `pcb` (per-channel-bin `(C,F2)`, time collapsed).

Four combinations; `sensefi+pcb` is the default (paper protocol), the others
(`sensefi+perpos`, `none+pcb`, `none+perpos`) are available for comparison.
Every distinct build gets its own bench dir named after `PRENORM`, `Z_GRAN` and
the UT-HAR `MERGE_VAL` flag, so runs never overwrite each other.

The config cell defaults reproduce the paper protocol with a single seed (42);
set `SEEDS = [0, 4, 8, 17, 42]` for the 5-seed statistics.

**Steps:** set `REPO_URL` in the clone cell, install the Mamba kernels + deps,
set `PRENORM`/`Z_GRAN` in the config cell, then run all cells.


In [ ]:
# Cell 2 — Clone / update the public code from GitHub
import sys, subprocess
from pathlib import Path

# EDIT before public release: point REPO_URL at the final public repository.
# Optionally pin REPO_REF to the paper release tag/commit for exact reproduction.
REPO_URL  = 'https://github.com/<owner>/wavmamba.git'
REPO_REF  = None          # e.g. 'v1.0' or a commit SHA; None = default branch HEAD
CODE_PATH = Path('/kaggle/working/wavmamba')

if not CODE_PATH.exists():
    subprocess.run(['git', 'clone', '--depth', '1', REPO_URL, str(CODE_PATH)], check=True)
else:
    subprocess.run(['git', 'pull'], cwd=str(CODE_PATH), check=True)
if REPO_REF:
    subprocess.run(['git', 'fetch', '--depth', '1', 'origin', REPO_REF],
                   cwd=str(CODE_PATH), check=True)
    subprocess.run(['git', 'checkout', REPO_REF], cwd=str(CODE_PATH), check=True)

sys.path.insert(0, str(CODE_PATH))
print(f'Code at {CODE_PATH}  (ref={REPO_REF or "HEAD"})')


In [ ]:
# Cell 2b — Install dependencies (run once per session, before any heavy import)
#
# mamba-ssm / causal-conv1d ship prebuilt CUDA wheels that must match the torch
# C++ ABI, so install them FIRST with --no-deps (so pip does not re-resolve or
# replace an ABI-compatible torch), then the regular requirements.
import subprocess, sys

def _pip(*args):
    subprocess.run([sys.executable, '-m', 'pip', 'install', *args], check=True)

# Kaggle images already ship a CUDA torch; skip the explicit torch pin there.
# Uncomment to force a specific torch build locally:
# _pip('torch==2.7.0')
_pip('mamba-ssm', '--no-build-isolation', '--no-deps')
_pip('causal-conv1d', '--no-deps')
_pip('-r', str(CODE_PATH / 'requirements.txt'))

from wavmamba import run, default_cfg, WavMamba, DIRMAP, bench_dirname, build_bench
print('Install + import OK : wavmamba.run')


In [ ]:
# Cell 3 — Configuration (defaults = the paper protocol)
from pathlib import Path

PRENORM   = 'sensefi'     # 'sensefi' = UT-HAR min-max / NTU-Fi (x-42.32)/4.98 | 'none' = no raw pre-normalization
Z_GRAN    = 'pcb'         # 'perpos' = z per-position (C,T2,F2) | 'pcb' = z per-channel-bin (C,F2), time collapsed
MERGE_VAL = True          # ONLY UT-HAR: True = merge val into test | False = test=X_test only
DATASETS  = ['uthar', 'ntufi']
SEEDS     = [42]          # default single seed; paper 5-seed protocol: [0, 4, 8, 17, 42]
OUT_ROOT  = '/kaggle/working'

_MARKER  = {'uthar': 'X_train.csv', 'ntufi': 'train_amp'}
assert PRENORM in ('none', 'sensefi'), f'bad PRENORM {PRENORM!r}'
assert Z_GRAN in ('perpos', 'pcb'),    f'bad Z_GRAN {Z_GRAN!r}'
# The bench/output tag is resolved per dataset (merge_val is UT-HAR only).
def run_tag(ds):
    return bench_dirname(PRENORM, Z_GRAN, MERGE_VAL and ds == 'uthar')

def resolve_mount(ds):
    base = Path('/kaggle/input')
    for c in (sorted(base.iterdir()) if base.is_dir() else []):
        if next(c.rglob(_MARKER[ds]), None) is not None:
            return str(c)
    raise FileNotFoundError(f'No /kaggle/input/* containing {_MARKER[ds]} for {ds}')

print(f'PRENORM={PRENORM}  Z_GRAN={Z_GRAN}  MERGE_VAL={MERGE_VAL}')
print(f'DATASETS={DATASETS}  SEEDS={SEEDS}')
for ds in DATASETS:
    try:    print(f'  {ds:6s} tag={run_tag(ds):16s} mount: {resolve_mount(ds)}')
    except Exception as e: print(f'  {ds:6s} !! {e}')


In [ ]:
# Cell 4 — Smoke: WavMamba builds + forwards on GPU (fail-fast before build/train)
import torch
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
# UT-HAR dims: n_antennas=3, f2=15 -> packed C = 2*3 = 6
_m = WavMamba(num_classes=7, n_antennas=3, f2=15).to(dev)
with torch.no_grad():
    _o = _m(torch.randn(2, 6, 125, 15, device=dev))   # T2=125 (250//2)
assert _o.shape == (2, 7), f'bad output {tuple(_o.shape)}'
print(f'SMOKE OK ({dev}) — WavMamba forward -> {tuple(_o.shape)}, '
      f'{sum(p.numel() for p in _m.parameters()):,} params')
del _m, _o
if dev == 'cuda': torch.cuda.empty_cache()


In [ ]:
# Cell 5 — Sweep: build + train each dataset (separate output dir per run)
import time, gc, torch

NL = chr(10)

def run_one(ds):
    raw   = resolve_mount(ds)
    # merge_val only exists for UT-HAR, so the tag is resolved PER DATASET —
    # bench_dirname() is the same helper build_bench() itself uses, so the
    # bench path and the output name always agree with the data.
    mv    = MERGE_VAL and ds == 'uthar'
    tag   = run_tag(ds)
    bench = Path(OUT_ROOT) / DIRMAP[ds] / 'bench' / tag
    out   = Path(f'{OUT_ROOT}/outputs/wavmamba_{ds}_{tag}')

    # Reuse an existing bench: /kaggle/working is wiped between sessions, but a
    # rerun inside one session should not pay the packing cost twice.
    if (bench / 'stats.json').exists():
        print(f'Bench exists, skipping build: {bench}')
    else:
        build_bench(ds, raw_root=raw, out_root=OUT_ROOT,
                    merge_val=mv, prenorm=PRENORM, z_gran=Z_GRAN)
    # run() reads classes / class names / dims from the bench's own stats.json.
    run(bench_dir=bench, output_dir=out, cfg=default_cfg(seeds=SEEDS),
        num_workers=4)
    return tag

results = {}
for ds in DATASETS:
    t0 = time.time()
    print(NL + '#' * 64 + NL + '#  wavmamba / ' + ds + ' / ' + run_tag(ds) + NL + '#' * 64)
    try:
        run_one(ds); results[ds] = 'OK'
    except Exception as e:
        results[ds] = f'FAILED: {type(e).__name__}: {e}'; print('!!', e)
    gc.collect()
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    print(f"== {ds}: {results[ds]}  ({(time.time()-t0)/60:.1f} min)")
print(NL + '=== SWEEP SUMMARY ===')
for k, v in results.items():
    print(f'  {k:14s}: {v}')

In [ ]:
# Cell 6 — Ablation study (optional): one-variable-at-a-time sweep
#
# Runs the 11 ablation variants for each dataset under the SAME protocol/flags
# as the main sweep above. Each variant builds the bench it needs (the DWT
# bench, or a separate raw_ bench for a1_raw), trains an AblationWavMamba, and
# is filed under outputs/ablation/<ds>/<variant>/. Finished variants are
# skipped, so this cell is resumable. Set ABLATE = False to skip it entirely.
import json, time, gc, torch
from wavmamba import ABLATIONS, ablation_table, build_ablation_model
from wavmamba.ablation import variant_front_end

NL       = chr(10)
ABLATE   = True                    # flip to False to skip the ablation sweep
VARIANTS = list(ABLATIONS)         # or a subset, e.g. ['ours', 'a1_raw', 'a4_bilstm']

def ablate_one(ds):
    for variant in VARIANTS:
        fe  = variant_front_end(variant)                       # 'dwt' | 'raw'
        mv  = MERGE_VAL and ds == 'uthar'
        tag = bench_dirname(PRENORM, Z_GRAN, mv, fe)
        bench = Path(OUT_ROOT) / DIRMAP[ds] / 'bench' / tag
        out   = Path(f'{OUT_ROOT}/outputs/ablation/{ds}/{variant}')

        mpath = out / 'metrics.json'
        if mpath.exists() and all(str(s) in json.load(open(mpath)).get('per_seed', {})
                                  for s in SEEDS):
            print(f'[skip] {variant}: already complete'); continue

        print(NL + '#' * 60 + NL + f'#  ablate / {ds} / {variant}  ({fe})' + NL + '#' * 60)
        if not (bench / 'stats.json').exists():
            build_bench(ds, raw_root=resolve_mount(ds), out_root=OUT_ROOT,
                        merge_val=mv, prenorm=PRENORM, z_gran=Z_GRAN, front_end=fe)
        meta = json.load(open(bench / 'stats.json'))['meta']
        run(bench_dir=bench, output_dir=out, cfg=default_cfg(seeds=SEEDS),
            num_workers=4,
            model_builder=lambda v=variant, m=meta: build_ablation_model(v, m))

if ABLATE:
    for ds in DATASETS:
        t0 = time.time()
        try:
            ablate_one(ds)
        except Exception as e:
            print('!!', type(e).__name__, e)
        gc.collect()
        if torch.cuda.is_available(): torch.cuda.empty_cache()
        print(f'== ablation {ds} done ({(time.time()-t0)/60:.1f} min)')
        print(NL + f'=== ABLATION TABLE — {ds} ===')
        ablation_table(ds, out_root=OUT_ROOT)


In [ ]:
# Cell 7 — Package the outputs into a single .zip for download
#
# /kaggle/working is already exposed as the notebook Output (a directory tree),
# but a single archive is easier to grab in one click. This zips everything
# under outputs/ (both the main sweep and the ablation runs, including each
# variant's metrics.json + seeds/ + the ablation summary.md / summary.csv).
import shutil
from pathlib import Path

src = Path(OUT_ROOT) / 'outputs'
if src.is_dir():
    archive = shutil.make_archive(str(Path(OUT_ROOT) / 'wavmamba_outputs'),
                                  'zip', root_dir=str(src))
    mb = Path(archive).stat().st_size / 1e6
    print(f'Zipped -> {archive}  ({mb:.1f} MB)')
else:
    print(f'Nothing to zip: {src} does not exist yet (run the sweep cells first).')
